In [1]:
# ================================================================
# AGE_INSURANCE DATASET
# K-MEANS / HIERARCHICAL / DBSCAN CLUSTERING
# ================================================================

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering,
    DBSCAN
)

from sklearn.metrics import silhouette_score

from scipy.cluster.hierarchy import (
    dendrogram,
    linkage
)


# ================================================================
# SETTINGS
# ================================================================

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)


print("=" * 70)
print("AGE_INSURANCE CUSTOMER CLUSTERING")
print("K-Means / Hierarchical / DBSCAN")
print("=" * 70)


# ================================================================
# 1. LOAD DATASET
# ================================================================

# Change this filename only if your actual CSV filename is different.
df = pd.read_csv("age_insurance (1).csv")


print("\nDataset loaded successfully.")

print("\nDataset shape:")
print(df.shape)


print("\nColumns:")
print(df.columns.tolist())


print("\nFirst 5 rows:")
print(df.head())


print("\nDataset information:")
print(df.info())


# ================================================================
# 2. CHECK MISSING VALUES
# ================================================================

print("\n" + "-" * 70)
print("MISSING VALUES")
print("-" * 70)

print(df.isnull().sum())


# ================================================================
# 3. DEFINE FEATURES
# ================================================================

FEATURE_COLS = [
    "Age",
    "Insurance",
    "InsurancePlan"
]


print("\nFeatures used for clustering:")
print(FEATURE_COLS)


# ================================================================
# 4. PREPROCESSING
# ================================================================

# Median imputation for missing numerical values
imputer = SimpleImputer(
    strategy="median"
)

feature_df = pd.DataFrame(
    imputer.fit_transform(
        df[FEATURE_COLS]
    ),
    columns=FEATURE_COLS,
    index=df.index
)


# Standardization
scaler = StandardScaler()

X = scaler.fit_transform(
    feature_df
)


print(
    f"\nNumber of customers: {len(df):,}"
)

print(
    f"Number of clustering features: {X.shape[1]}"
)


# ================================================================
# 5. K-MEANS: ELBOW METHOD + SILHOUETTE SCORE
# ================================================================

print("\n" + "-" * 70)
print("K-MEANS: FINDING OPTIMAL NUMBER OF CLUSTERS")
print("-" * 70)


k_range = range(2, 9)

inertias = []

sil_scores = []


for k in k_range:

    km = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=RANDOM_STATE
    )

    labels = km.fit_predict(X)

    # Elbow / inertia
    inertias.append(
        km.inertia_
    )

    # Silhouette
    sil_scores.append(
        silhouette_score(
            X,
            labels
        )
    )


# ================================================================
# 6. PLOT ELBOW + SILHOUETTE
# ================================================================

fig, (ax1, ax2) = plt.subplots(
    1,
    2,
    figsize=(12, 4.5)
)


# Elbow plot
ax1.plot(
    list(k_range),
    inertias,
    marker="o"
)

ax1.set_title(
    "Elbow Method"
)

ax1.set_xlabel(
    "Number of Clusters (k)"
)

ax1.set_ylabel(
    "Inertia"
)


# Silhouette plot
ax2.plot(
    list(k_range),
    sil_scores,
    marker="o"
)

ax2.set_title(
    "Silhouette Score vs k"
)

ax2.set_xlabel(
    "Number of Clusters (k)"
)

ax2.set_ylabel(
    "Silhouette Score"
)


plt.tight_layout()


plt.savefig(
    "age_insurance_elbow_silhouette.png",
    dpi=150
)


plt.close(fig)


print(
    "\nSaved plot:"
)

print(
    "age_insurance_elbow_silhouette.png"
)


# ================================================================
# 7. DISPLAY SILHOUETTE SCORES
# ================================================================

print("\nSilhouette scores:")


for k, score in zip(
    k_range,
    sil_scores
):

    print(
        f"k = {k}: {score:.3f}"
    )


# Select k having highest silhouette score
best_k = list(k_range)[
    int(
        np.argmax(
            sil_scores
        )
    )
]


print(
    f"\nSilhouette-suggested k = {best_k}"
)


# ================================================================
# 8. K-MEANS CLUSTERING
# ================================================================

print("\n" + "-" * 70)
print("K-MEANS CLUSTERING")
print("-" * 70)


kmeans = KMeans(
    n_clusters=best_k,
    n_init=10,
    random_state=RANDOM_STATE
)


km_labels = kmeans.fit_predict(
    X
)


# Silhouette score
km_silhouette = silhouette_score(
    X,
    km_labels
)


print(
    f"K-Means silhouette score: "
    f"{km_silhouette:.3f}"
)


# Cluster sizes
cluster_sizes = (
    pd.Series(km_labels)
    .value_counts()
    .sort_index()
)


print("\nCluster sizes:")

print(
    cluster_sizes.to_dict()
)


# ================================================================
# 9. K-MEANS CLUSTER PROFILING
# ================================================================

profiled = feature_df.copy()

profiled[
    "KMeansCluster"
] = km_labels


summary = profiled.groupby(
    "KMeansCluster"
).agg(

    NumberOfCustomers=(
        "Age",
        "size"
    ),

    AverageAge=(
        "Age",
        "mean"
    ),

    AverageInsurance=(
        "Insurance",
        "mean"
    ),

    AverageInsurancePlan=(
        "InsurancePlan",
        "mean"
    )

).round(3)


print("\n" + "-" * 70)
print("K-MEANS CLUSTER PROFILES")
print("-" * 70)


print(
    summary.to_string()
)


# Save cluster profile
summary.to_csv(
    "age_insurance_cluster_profiles.csv"
)


print(
    "\nSaved:"
)

print(
    "age_insurance_cluster_profiles.csv"
)


# ================================================================
# 10. HIERARCHICAL CLUSTERING
# ================================================================

print("\n" + "-" * 70)
print("HIERARCHICAL CLUSTERING")
print("-" * 70)


hierarchical = AgglomerativeClustering(
    n_clusters=best_k,
    linkage="ward"
)


hc_labels = hierarchical.fit_predict(
    X
)


hc_silhouette = silhouette_score(
    X,
    hc_labels
)


print(
    f"Hierarchical silhouette score: "
    f"{hc_silhouette:.3f}"
)


print(
    "\nHierarchical cluster sizes:"
)


print(
    pd.Series(
        hc_labels
    )
    .value_counts()
    .sort_index()
    .to_dict()
)


# ================================================================
# 11. DBSCAN
# ================================================================

print("\n" + "-" * 70)
print("DBSCAN CLUSTERING")
print("-" * 70)


# You can experiment with eps and min_samples.
dbscan = DBSCAN(
    eps=0.8,
    min_samples=8
)


db_labels = dbscan.fit_predict(
    X
)


# Number of clusters
n_db_clusters = (
    len(set(db_labels))
    -
    (1 if -1 in db_labels else 0)
)


# Number of noise points
n_noise = int(
    np.sum(
        db_labels == -1
    )
)


print(
    f"DBSCAN clusters: "
    f"{n_db_clusters}"
)


print(
    f"DBSCAN noise points: "
    f"{n_noise}"
)


# Calculate DBSCAN silhouette
if n_db_clusters >= 2:

    mask = (
        db_labels != -1
    )

    if len(
        set(
            db_labels[mask]
        )
    ) >= 2:

        db_silhouette = silhouette_score(
            X[mask],
            db_labels[mask]
        )

        print(
            f"DBSCAN silhouette "
            f"(excluding noise): "
            f"{db_silhouette:.3f}"
        )

    else:

        db_silhouette = None

        print(
            "DBSCAN silhouette cannot be calculated."
        )

else:

    db_silhouette = None

    print(
        "DBSCAN produced fewer than 2 clusters."
    )


# ================================================================
# 12. HIERARCHICAL DENDROGRAM
# ================================================================

print("\n" + "-" * 70)
print("CREATING DENDROGRAM")
print("-" * 70)


Z = linkage(
    X,
    method="ward"
)


plt.figure(
    figsize=(12, 6)
)


dendrogram(
    Z,
    truncate_mode="lastp",
    p=30
)


plt.title(
    "Hierarchical Clustering Dendrogram"
)

plt.xlabel(
    "Cluster / Sample"
)

plt.ylabel(
    "Ward Distance"
)


plt.tight_layout()


plt.savefig(
    "age_insurance_dendrogram.png",
    dpi=150
)


plt.close()


print(
    "Saved:"
)

print(
    "age_insurance_dendrogram.png"
)


# ================================================================
# 13. PCA
# ================================================================

print("\n" + "-" * 70)
print("PCA VISUALIZATION")
print("-" * 70)


pca = PCA(
    n_components=2,
    random_state=RANDOM_STATE
)


X_pca = pca.fit_transform(
    X
)


pc1_variance = (
    pca.explained_variance_ratio_[0]
)


pc2_variance = (
    pca.explained_variance_ratio_[1]
)


print(
    f"PC1 explained variance: "
    f"{pc1_variance:.3f}"
)


print(
    f"PC2 explained variance: "
    f"{pc2_variance:.3f}"
)


print(
    f"Total explained variance: "
    f"{pc1_variance + pc2_variance:.3f}"
)


# ================================================================
# 14. K-MEANS PCA VISUALIZATION
# ================================================================

plt.figure(
    figsize=(7, 5.5)
)


plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=km_labels,
    cmap="tab10",
    s=35,
    alpha=0.7
)


plt.title(
    f"K-Means Customer Clusters "
    f"({best_k} clusters)"
)


plt.xlabel(
    "Principal Component 1"
)

plt.ylabel(
    "Principal Component 2"
)


plt.tight_layout()


plt.savefig(
    "age_insurance_kmeans_pca.png",
    dpi=150
)


plt.close()


print(
    "Saved:"
)

print(
    "age_insurance_kmeans_pca.png"
)


# ================================================================
# 15. HIERARCHICAL + DBSCAN PCA VISUALIZATION
# ================================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5)
)


# ------------------------------------------------
# Hierarchical
# ------------------------------------------------

axes[0].scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=hc_labels,
    cmap="tab10",
    s=35,
    alpha=0.7
)


axes[0].set_title(
    f"Hierarchical Clustering "
    f"({best_k} clusters)"
)


axes[0].set_xlabel(
    "Principal Component 1"
)

axes[0].set_ylabel(
    "Principal Component 2"
)


# ------------------------------------------------
# DBSCAN
# ------------------------------------------------

noise_mask = (
    db_labels == -1
)


# Normal DBSCAN clusters
if np.any(
    ~noise_mask
):

    axes[1].scatter(
        X_pca[
            ~noise_mask,
            0
        ],
        X_pca[
            ~noise_mask,
            1
        ],
        c=db_labels[
            ~noise_mask
        ],
        cmap="tab10",
        s=35,
        alpha=0.7
    )


# Noise points
if np.any(
    noise_mask
):

    axes[1].scatter(
        X_pca[
            noise_mask,
            0
        ],
        X_pca[
            noise_mask,
            1
        ],
        c="lightgray",
        s=35,
        marker="x",
        label="Noise"
    )

    axes[1].legend(
        loc="upper right"
    )


axes[1].set_title(
    f"DBSCAN "
    f"({n_db_clusters} clusters)"
)


axes[1].set_xlabel(
    "Principal Component 1"
)

axes[1].set_ylabel(
    "Principal Component 2"
)


plt.tight_layout()


plt.savefig(
    "age_insurance_hierarchical_dbscan_pca.png",
    dpi=150
)


plt.close(fig)


print(
    "Saved:"
)

print(
    "age_insurance_hierarchical_dbscan_pca.png"
)


# ================================================================
# 16. ADD K-MEANS CLUSTER TO ORIGINAL DATA
# ================================================================

df_out = df.copy()


df_out[
    "KMeansCluster"
] = km_labels


df_out.to_csv(
    "age_insurance_with_clusters.csv",
    index=False
)


print("\n" + "-" * 70)

print(
    "FINAL DATASET SAVED"
)

print("-" * 70)


print(
    "age_insurance_with_clusters.csv"
)


# ================================================================
# 17. METHOD COMPARISON
# ================================================================

print("\n" + "=" * 70)

print(
    "CLUSTERING METHOD COMPARISON"
)

print("=" * 70)


print(
    f"K-Means silhouette:       "
    f"{km_silhouette:.3f}"
)


print(
    f"Hierarchical silhouette:  "
    f"{hc_silhouette:.3f}"
)


if db_silhouette is not None:

    print(
        f"DBSCAN silhouette:        "
        f"{db_silhouette:.3f}"
    )

else:

    print(
        "DBSCAN silhouette:        "
        "Not available"
    )


print("\n" + "=" * 70)

print(
    "GENERATED FILES"
)

print("=" * 70)


print(
    "1. age_insurance_elbow_silhouette.png"
)

print(
    "2. age_insurance_cluster_profiles.csv"
)

print(
    "3. age_insurance_dendrogram.png"
)

print(
    "4. age_insurance_kmeans_pca.png"
)

print(
    "5. age_insurance_hierarchical_dbscan_pca.png"
)

print(
    "6. age_insurance_with_clusters.csv"
)


print("\nDone!")

AGE_INSURANCE CUSTOMER CLUSTERING
K-Means / Hierarchical / DBSCAN

Dataset loaded successfully.

Dataset shape:
(300, 3)

Columns:
['Age', 'Insurance', 'InsurancePlan']

First 5 rows:
   Age  Insurance  InsurancePlan
0   56          0              0
1   69          1              2
2   46          1              2
3   32          0              0
4   60          1              2

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Age            300 non-null    int64
 1   Insurance      300 non-null    int64
 2   InsurancePlan  300 non-null    int64
dtypes: int64(3)
memory usage: 7.2 KB
None

----------------------------------------------------------------------
MISSING VALUES
----------------------------------------------------------------------
Age              0
Insurance        0
InsurancePlan    0
dtype: int64

Features u